In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install mlflow
import mlflow
import mlflow.sklearn
mlflow.set_experiment("credit_risk_classification")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1772125955265, experiment_id='1', last_update_time=1772125955265, lifecycle_stage='active', name='credit_risk_classification', tags={}, workspace='default'>

In [ ]:
#LIBRARIES
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
!pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import joblib
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
#Load the data
X_train=pd.read_csv('/content/drive/MyDrive/Data/X_train.csv')
X_test=pd.read_csv('/content/drive/MyDrive/Data/X_test.csv')
y_train = pd.read_csv('/content/drive/MyDrive/Data/y_train.csv')
y_test = pd.read_csv('/content/drive/MyDrive/Data/y_test.csv')

In [ ]:
#LOGISTIC REGRESSION(BASELINE)

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))])
lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)
#evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Logistic Regression Performance:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
#Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.savefig('/content/drive/MyDrive/T2_Project_ShruthiA/Visualizations/model_cm.png')
plt.close()
#ML Flow
with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
  # Log model parameters
  mlflow.log_param("model_type", "Logistic Regression")
  lr_model = lr_pipeline.named_steps['classifier']
  mlflow.log_param("random_state", 42)
  mlflow.log_param("penalty", lr_model.penalty)
  mlflow.log_param("C", lr_model.C)
  mlflow.log_param("solver", lr_model.solver)
  # Log metrics
  mlflow.log_metric("accuracy", accuracy)
  mlflow.log_metric("precision", precision)
  mlflow.log_metric("recall", recall)
  mlflow.log_metric("f1_score", f1)
  # Log confusion matrix artifact
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/model_cm.png")
  # Log trained model
  mlflow.sklearn.log_model(lr_pipeline, name="model")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Logistic Regression Performance:
Accuracy : 0.72
Precision: 0.676923076923077
Recall   : 0.5569620253164557
F1 Score : 0.6111111111111112


2026/02/26 17:19:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#DECISION TREE
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
# Metrics
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
print("Decision Tree Performance:")
print("Accuracy :", accuracy_dt)
print("Precision:", precision_dt)
print("Recall   :", recall_dt)
print("F1 Score :", f1_dt)

#Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt)
disp.plot()
plt.title("Decision Tree Confusion Matrix")
plt.savefig("/content/drive/MyDrive/Visualizations/dt_cm.png")
plt.close()
#ML Flow
with mlflow.start_run(run_name="Decision_tree"):
#Log model parameters
  for param, value in dt_model.get_params().items():
    mlflow.log_param(param, value)
#Log metrics
  mlflow.log_metric("accuracy", accuracy_dt)
  mlflow.log_metric("precision", precision_dt)
  mlflow.log_metric("recall", recall_dt)
  mlflow.log_metric("f1_score", f1_dt)
#Log confusion matrix
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/dt_cm.png")
#Log trained model
  mlflow.sklearn.log_model(dt_model, name="model")

Decision Tree Performance:
Accuracy : 0.79
Precision: 0.7283950617283951
Recall   : 0.7468354430379747
F1 Score : 0.7375


2026/02/26 17:19:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#RANDOM FOREST
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
# Metrics
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
print("Random Forest Performance:")
print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)
#Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.savefig("/content/drive/MyDrive/Visualizations/rf_cm.png")
plt.close()
#ML Flow
with mlflow.start_run(run_name="Random_Forest"):
# Log model parameters
  for param, value in rf_model.get_params().items():
      mlflow.log_param(param, value)
# Log metrics
  mlflow.log_metric("accuracy", accuracy_rf)
  mlflow.log_metric("precision", precision_rf)
  mlflow.log_metric("recall", recall_rf)
  mlflow.log_metric("f1_score", f1_rf)
  #Log confusion matrix artifact
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/rf_cm.png")
# Log trained model
  mlflow.sklearn.log_model(rf_model, name="model")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest Performance:
Accuracy : 0.81
Precision: 0.8059701492537313
Recall   : 0.6835443037974683
F1 Score : 0.7397260273972602


2026/02/26 17:20:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#XGBoost
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
# Metrics
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
print("XGBoost Performance:")
print("Accuracy :", accuracy_xgb)
print("Precision:", precision_xgb)
print("Recall   :", recall_xgb)
print("F1 Score :", f1_xgb)
#Confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb)
disp.plot()
plt.title("XGBoost Confusion Matrix")
plt.savefig("/content/drive/MyDrive/Visualizations/xgb_cm.png")
plt.close()
#ML FLow
with mlflow.start_run(run_name="XGB_Boost"):
#Log model parameters
  for param, value in xgb_model.get_params().items():
    mlflow.log_param(param, value)
#Log metrics
  mlflow.log_metric("accuracy", accuracy_xgb)
  mlflow.log_metric("precision", precision_xgb)
  mlflow.log_metric("recall", recall_xgb)
  mlflow.log_metric("f1_score", f1_xgb)
 #Log confusion matrix artifact
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/xgb_cm.png")
#Log trained model
  mlflow.sklearn.log_model(xgb_model, name="model")


XGBoost Performance:
Accuracy : 0.85
Precision: 0.8266666666666667
Recall   : 0.7848101265822784
F1 Score : 0.8051948051948052


2026/02/26 17:20:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#HYPERPARAMETER TUNING
#1.Random Forest GridSearchCV

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,cv=5,scoring='f1',n_jobs=-1,verbose=1)
grid_rf.fit(X_train, y_train)
print("Best RF Params:", grid_rf.best_params_)
print("Best RF CV Score:", grid_rf.best_score_)
best_rf = grid_rf.best_estimator_

#Evaluate tuned RF
y_pred_best_rf = best_rf.predict(X_test)
#metrics
accuracy_rf_tuned = accuracy_score(y_test, y_pred_best_rf)
precision_rf_tuned = precision_score(y_test, y_pred_best_rf)
recall_rf_tuned = recall_score(y_test, y_pred_best_rf)
f1_rf_tuned = f1_score(y_test, y_pred_best_rf)
print("Tuned Random Forest Performance:")
print("Accuracy :", accuracy_rf_tuned)
print("Precision:", precision_rf_tuned)
print("Recall   :", recall_rf_tuned)
print("F1 Score :", f1_rf_tuned)
#Confusion matrix
cm_rf_tuned = confusion_matrix(y_test, y_pred_best_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf_tuned)
disp.plot()
plt.title("Tuned RF Confusion Matrix")
plt.savefig("/content/drive/MyDrive/Visualizations/rf_tuned_cm.png")
plt.close()
#MLflow for Tuned RF
with mlflow.start_run(run_name="Random_Forest_Tuned"):
#Log hyperparameters
  for param, value in grid_rf.best_params_.items():
      mlflow.log_param(param, value)
#Log metrics
  mlflow.log_metric("accuracy", accuracy_rf_tuned)
  mlflow.log_metric("precision", precision_rf_tuned)
  mlflow.log_metric("recall", recall_rf_tuned)
  mlflow.log_metric("f1_score", f1_rf_tuned)
#Log confusion matrix
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/rf_tuned_cm.png")
#Log model
  mlflow.sklearn.log_model(best_rf, name="model")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best RF Params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best RF CV Score: 0.7995939989895905
Tuned Random Forest Performance:
Accuracy : 0.795
Precision: 0.7878787878787878
Recall   : 0.6582278481012658
F1 Score : 0.7172413793103448


2026/02/26 17:21:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#2.XGBoost RandommizedSearch(HYPERTUNING)
param_dist_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0]}
random_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_dist_xgb,n_iter=10,cv=5,scoring='f1',n_jobs=-1,
    verbose=1)
random_xgb.fit(X_train, y_train)
print("Best XGB Params:", random_xgb.best_params_)
print("Best XGB CV Score:", random_xgb.best_score_)
best_xgb = random_xgb.best_estimator_

#Evaluate Tuned XGB
y_pred_best_xgb = best_xgb.predict(X_test)
#Metrics
accuracy_xgb_tuned = accuracy_score(y_test, y_pred_best_xgb)
precision_xgb_tuned = precision_score(y_test, y_pred_best_xgb)
recall_xgb_tuned = recall_score(y_test, y_pred_best_xgb)
f1_xgb_tuned = f1_score(y_test, y_pred_best_xgb)
print("Tuned XGBoost Performance:")
print("Accuracy :", accuracy_xgb_tuned)
print("Precision:", precision_xgb_tuned)
print("Recall   :", recall_xgb_tuned)
print("F1 Score :", f1_xgb_tuned)
#Confusion Matrix
cm_xgb_tuned = confusion_matrix(y_test, y_pred_best_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_xgb_tuned)
disp.plot()
plt.title("Tuned XGB Confusion Matrix")
plt.savefig("/content/drive/MyDrive/Visualizations/xgb_tuned_cm.png")
plt.close()
#MLflow for Tuned XGB
with mlflow.start_run(run_name="XGBoost_Tuned"):
#Log hyperparameters
  for param, value in random_xgb.best_params_.items():
    mlflow.log_param(param, value)
#Log metrics
  mlflow.log_metric("accuracy", accuracy_xgb_tuned)
  mlflow.log_metric("precision", precision_xgb_tuned)
  mlflow.log_metric("recall", recall_xgb_tuned)
  mlflow.log_metric("f1_score", f1_xgb_tuned)
#Log confusion matrix
  mlflow.log_artifact("/content/drive/MyDrive/Visualizations/xgb_tuned_cm.png")
#Log model
  mlflow.sklearn.log_model(best_xgb,name="model")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best XGB Params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}
Best XGB CV Score: 0.8240461133562841
Tuned XGBoost Performance:
Accuracy : 0.86
Precision: 0.8493150684931506
Recall   : 0.7848101265822784
F1 Score : 0.8157894736842105


2026/02/26 17:22:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
import mlflow

runs = mlflow.search_runs()
print(runs.columns.tolist())  # shows all available columns
print(runs)

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.recall', 'metrics.f1_score', 'metrics.precision', 'metrics.accuracy', 'params.subsample', 'params.n_estimators', 'params.learning_rate', 'params.max_depth', 'params.min_samples_split', 'params.min_samples_leaf', 'params.interaction_constraints', 'params.feature_weights', 'params.callbacks', 'params.feature_types', 'params.max_delta_step', 'params.random_state', 'params.colsample_bylevel', 'params.min_child_weight', 'params.n_jobs', 'params.base_score', 'params.importance_type', 'params.max_bin', 'params.device', 'params.verbosity', 'params.max_leaves', 'params.enable_categorical', 'params.missing', 'params.objective', 'params.monotone_constraints', 'params.reg_alpha', 'params.eval_metric', 'params.reg_lambda', 'params.tree_method', 'params.booster', 'params.sampling_method', 'params.colsample_bytree', 'params.early_stopping_rounds', 'params.colsample_bynode', 'params.max_cat_threshold', 'params.gro

### 📊 Model Comparison

| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|----------|--------|----|
| Logistic Regression | 0.72 | 0.6769 | 0.5569 | 0.6111|
| Decision Tree | 0.79 | 0.7283 | 0.7468 | 0.7375 |
| Random Forest | 0.81 | 0.8059 | 0.6835 | 0.7397 |
| Random Forest Tuned | 0.795 | 0.7878 | 0.6582 | 0.7172 |
| XGBoost | 0.85 | 0.8266 | 0.7848 | 0.8051 |
| XGBoost Tuned | 0.855 | 0.8289| 0.7974 | 0.8129 |

Observation:

**1.XGBoost Tuned** performed the best overall with the highest Accuracy (0.855) and F1 Score (0.8129). It provides the best balance between Precision and Recall.

**2.Random Forest tuning** did not improve performance. The F1 score decreased slightly from 0.7397 to 0.7172, which shows that tuning does not always guarantee better results on test data.

**3.Decision Tree** and **Random Forest** performed better than **Logistic Regression**, showing that tree-based models handle this dataset more effectively.

**4.Logistic Regression** serves as a basic baseline
model but has lower Recall and F1 compared to ensemble models.

**CONCLUSION**

**Tuned XGBoost** is the best-performing model for this classification problem, achieving the highest Accuracy and F1 Score.

In [ ]:
model_folder = "/content/drive/MyDrive/Models"

# Save models
joblib.dump(lr_pipeline, f"{model_folder}/logistic_regression.pkl")
joblib.dump(dt_model, f"{model_folder}/decision_tree.pkl")
joblib.dump(rf_model, f"{model_folder}/random_forest.pkl")
joblib.dump(best_rf, f"{model_folder}/random_forest_tuned.pkl")
joblib.dump(xgb_model, f"{model_folder}/xgboost.pkl")
joblib.dump(best_xgb, f"{model_folder}/xgboost_tuned.pkl")

print("All models saved successfully")

All models saved successfully
